# Experimento D: Interpretabilidad y Caja Blanca (XAI)

## Objetivo
Cumplir el Objetivo Específico 6: **"Analizar la importancia de las características"**.  
No basta con decir "es fraude", hay que decir **por qué**.

## Técnicas

| Técnica | Método | Entregable |
|---------|--------|------------|
| **Feature Importance Nativa** | `feature_importances_` (Gain) de XGBoost | Gráfico de barras Top-10 variables |
| **SHAP Values (global)** | `TreeExplainer` sobre muestra de test | Beeswarm plot |
| **SHAP Values (local)** | Force plots individuales | 2 Force plots: fraude vs normal |

## Modelo Utilizado
**XGBoost cost-sensitive** (`scale_pos_weight` dinámico) entrenado con división temporal estricta.  
Se comparan las Feature Importances entre el modelo Baseline (Exp A) y el cost-sensitive.

## Métricas del Modelo
- AUPRC, AUC ROC, Card Precision@100

In [ ]:
import os
import sys
import warnings
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn import metrics
import xgboost as xgb
import shap

# Configuración del proyecto
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))

from experiments.config import (
    SEED, INPUT_FEATURES, OUTPUT_FEATURE,
    COST_SENSITIVE_PARAMS, RESULTS_DIR, FIGURES_DIR, COLORS, TOP_K_LIST,
    START_DATE_TRAINING, DELTA_TRAIN, DELTA_DELAY, DELTA_TEST,
)
from experiments.data_utils import (
    load_transformed_data, get_train_test_set,
    print_dataset_summary, compute_class_ratio,
    card_precision_top_k,
)

warnings.filterwarnings('ignore')
sns.set_style('darkgrid', {'axes.facecolor': '0.9'})

print("=" * 60)
print("  EXPERIMENTO D: INTERPRETABILIDAD Y XAI")
print("=" * 60)
print(f"  Semilla: {SEED}")
print(f"  Modelo: XGBoost cost-sensitive")
print(f"  Técnicas: Feature Importance (Gain) + SHAP Values")

---
## 1. Preparación de Datos y Modelo

In [ ]:
transactions_df = load_transformed_data()
train_df, test_df = get_train_test_set(
    transactions_df,
    start_date_training=START_DATE_TRAINING,
    delta_train=DELTA_TRAIN,
    delta_delay=DELTA_DELAY,
    delta_test=DELTA_TEST,
)
print_dataset_summary(train_df, test_df, "Experimento D - Interpretabilidad")

# Entrenar modelo XGBoost cost-sensitive
scale_pos_weight = compute_class_ratio(train_df[OUTPUT_FEATURE])
xgb_params = {**COST_SENSITIVE_PARAMS["XGBoost"], "scale_pos_weight": scale_pos_weight}
print(f"\nscale_pos_weight: {scale_pos_weight:.2f}")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(train_df[INPUT_FEATURES])
X_test_scaled = scaler.transform(test_df[INPUT_FEATURES])

model_xgb = xgb.XGBClassifier(**xgb_params)
model_xgb.fit(X_train_scaled, train_df[OUTPUT_FEATURE])

# Métricas del modelo
y_pred_proba = model_xgb.predict_proba(X_test_scaled)[:, 1]
auprc = metrics.average_precision_score(test_df[OUTPUT_FEATURE], y_pred_proba)
auc_roc = metrics.roc_auc_score(test_df[OUTPUT_FEATURE], y_pred_proba)

predictions_d_df = test_df.copy()
predictions_d_df['predictions'] = y_pred_proba
_, _, cp100 = card_precision_top_k(predictions_d_df, top_k=100)

print(f"\n  XGBoost cost-sensitive entrenado:")
print(f"    AUC ROC:  {auc_roc:.4f}")
print(f"    AUPRC:    {auprc:.4f}")
print(f"    CP@100:   {cp100:.4f}")

---
## 2. Feature Importance Nativa (Gini/Gain)

In [ ]:
# Extraer feature importances nativas de XGBoost
importances = model_xgb.feature_importances_
feature_importance_df = pd.DataFrame({
    'Feature': INPUT_FEATURES,
    'Importance': importances,
}).sort_values('Importance', ascending=False)

# Top 10 variables
top_10 = feature_importance_df.head(10)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(range(len(top_10)), top_10['Importance'].values, color=COLORS['baseline'])
ax.set_yticks(range(len(top_10)))
ax.set_yticklabels(top_10['Feature'].values, fontsize=11)
ax.invert_yaxis()
ax.set_xlabel('Importancia (Gain)', fontsize=12)
ax.set_title('Top-10 Variables Más Importantes\n(XGBoost Cost-Sensitive)', fontsize=14)

for i, (val, name) in enumerate(zip(top_10['Importance'], top_10['Feature'])):
    ax.text(val + 0.002, i, f'{val:.4f}', va='center', fontsize=10)

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'experiment_d_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nTop-10 variables:")
display(top_10.reset_index(drop=True))

---
## 3. SHAP Values - Análisis Global

In [ ]:
# Calcular SHAP values sobre una muestra del test set
# (usar muestra para eficiencia computacional)
SAMPLE_SIZE = min(500, len(X_test_scaled))
np.random.seed(SEED)
sample_indices = np.random.choice(len(X_test_scaled), SAMPLE_SIZE, replace=False)
X_sample = X_test_scaled[sample_indices]

explainer = shap.TreeExplainer(model_xgb)
shap_values = explainer.shap_values(X_sample)

# Crear DataFrame con nombres de features para mejor visualización
X_sample_df = pd.DataFrame(X_sample, columns=INPUT_FEATURES)

print(f"SHAP values calculados para {SAMPLE_SIZE} muestras del test set")

In [ ]:
# Beeswarm plot (resumen global de importancia SHAP)
fig = plt.figure(figsize=(12, 7))
shap.summary_plot(shap_values, X_sample_df, show=False)
plt.title('SHAP Beeswarm Plot - XGBoost Cost-Sensitive\n', fontsize=14)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'experiment_d_shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()
print("Beeswarm plot generado")

---
## 4. SHAP Force Plots - Explicaciones Individuales

In [ ]:
# Encontrar ejemplos de fraude y transacción normal en la muestra
y_test_sample = test_df[OUTPUT_FEATURE].iloc[sample_indices].values
preds_sample = model_xgb.predict_proba(X_sample)[:, 1]

# Transacción FRAUDULENTA con mayor probabilidad
fraud_indices = np.where(y_test_sample == 1)[0]
if len(fraud_indices) > 0:
    fraud_idx = fraud_indices[np.argmax(preds_sample[fraud_indices])]
    print(f"Ejemplo FRAUDE (índice {fraud_idx}): probabilidad = {preds_sample[fraud_idx]:.4f}")
    
    # Force plot para fraude (como matplotlib figure)
    fig = plt.figure(figsize=(16, 3))
    shap.force_plot(
        explainer.expected_value, shap_values[fraud_idx],
        X_sample_df.iloc[fraud_idx], matplotlib=True, show=False
    )
    plt.title('Force Plot - Transacción FRAUDULENTA', fontsize=12)
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / 'experiment_d_shap_force_fraud.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("⚠ No se encontraron fraudes en la muestra")

# Transacción NORMAL con menor probabilidad de fraude
normal_indices = np.where(y_test_sample == 0)[0]
normal_idx = normal_indices[np.argmin(preds_sample[normal_indices])]
print(f"\nEjemplo NORMAL (índice {normal_idx}): probabilidad = {preds_sample[normal_idx]:.4f}")

fig = plt.figure(figsize=(16, 3))
shap.force_plot(
    explainer.expected_value, shap_values[normal_idx],
    X_sample_df.iloc[normal_idx], matplotlib=True, show=False
)
plt.title('Force Plot - Transacción NORMAL', fontsize=12)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'experiment_d_shap_force_normal.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Guardar Resultados

In [ ]:
# Guardar resultados
feature_importance_df.to_csv(RESULTS_DIR / 'experiment_d_feature_importance.csv', index=False)

results_d = {
    'feature_importance': feature_importance_df,
    'shap_values': shap_values,
    'X_sample': X_sample_df,
    'metrics': {
        'auc_roc': auc_roc,
        'auprc': auprc,
        'card_precision_at_100': cp100,
    },
    'metadata': {
        'model': 'XGBoost cost-sensitive',
        'scale_pos_weight': scale_pos_weight,
        'seed': SEED,
        'shap_sample_size': SAMPLE_SIZE,
    },
}
with open(RESULTS_DIR / 'experiment_d_results.pkl', 'wb') as f:
    pickle.dump(results_d, f)

print("✓ Resultados del Experimento D guardados exitosamente")
print(f"  - Feature importance: {RESULTS_DIR / 'experiment_d_feature_importance.csv'}")
print(f"  - Resultados PKL: {RESULTS_DIR / 'experiment_d_results.pkl'}")
print(f"  - Figuras: {FIGURES_DIR}")

---
## 6. Conclusiones del Experimento D

**Entregables generados:**
1. Gráfico de barras con Top-10 variables (Feature Importance nativa)
2. Beeswarm plot SHAP (resumen global de la dirección de cada variable)
3. Force plots individuales (ejemplo de fraude vs transacción normal)

Estos entregables demuestran no solo **qué** variables son importantes, sino **en qué dirección** influyen en la predicción de fraude.